# E1 Compression Breakout — Backtesting Notebook

V7.1 spec. Phase 1: BTC-USDT only.

**Walk-forward split (LOCKED):**
- Train: 2023-01-01 → 2024-06-30
- Validation: 2024-07-01 → 2025-01-01 (Sharpe 1.29, adj PnL $76)
- Out-of-sample: 2025-01-01 → today — DO NOT RUN until ready to go live

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, '/Users/hermes/quants-lab')

from core.backtesting import BacktestingEngine
import datetime

backtesting = BacktestingEngine(load_cached_data=True)
print("BacktestingEngine ready")

## Strategy Config

Locked params. Do not change between stress test runs.

In [ ]:
from app.controllers.directional_trading.e1_compression_breakout import (
    E1CompressionBreakoutConfig,
)
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from hummingbot.core.data_type.common import OrderType
from decimal import Decimal

# ---- LOCKED PARAMS (do not tune) ----
config = E1CompressionBreakoutConfig(
    id="e1_backtest_001",
    connector_name="binance_perpetual",  # DATA SOURCE for backtesting only — switch to "bybit_perpetual" for live deployment
    trading_pair="BTC-USDT",
    total_amount_quote=Decimal("1000"),
    max_executors_per_side=1,
    cooldown_time=300,
    leverage=1,
    # E1 signal params
    atr_period=14,
    atr_compression_window=90  # unit: days,
    atr_compression_threshold=0.20,
    range_period=20,
    volume_period=20,
    volume_floor_multiplier=1.3,
    volume_spike_multiplier=2.0,
    risk_off_ema_period=20,
    risk_off_threshold=0.97,
    # Triple barrier — flat fields (this version of hummingbot)
    stop_loss=Decimal("0.015"),
    take_profit=Decimal("0.03"),
    time_limit=86400,
    trailing_stop=TrailingStop(
        activation_price=Decimal("0.015"),
        trailing_delta=Decimal("0.005"),
    ),
    take_profit_order_type=OrderType.MARKET,
)
print(f"Config ready: {config.controller_name} | {config.trading_pair} | SL {config.stop_loss} | TP {config.take_profit}")

## Run Backtest

Change `start`/`end` for different windows. Keep `trade_cost=0.000375` (0.02% maker in + 0.055% taker out).

In [ ]:
# ---- STRESS TEST WINDOWS (pick one) ----
# Bear Crash:       2021-11-01 → 2022-06-30
# FTX Shock:        2022-11-01 → 2022-11-30
# Low-Vol Ranging:  2023-01-01 → 2023-09-30
# Bull Validation:  2024-07-01 → 2025-01-01  ← LOCKED reference
# OUT-OF-SAMPLE:    2025-01-01 → today        ← DO NOT RUN

start = int(datetime.datetime(2024, 7, 1).timestamp())
end   = int(datetime.datetime(2025, 1, 1).timestamp())
trade_cost = 0.000375  # 0.02% maker in + 0.055% taker out = 0.075% RT

backtesting_result = await backtesting.run_backtesting(
    config=config,
    start=start,
    end=end,
    backtesting_resolution="1m",
    trade_cost=trade_cost,
)
# Workaround: library bug — close_types returns int(0) when no trades (should be {})
if not isinstance(backtesting_result.results.get("close_types"), dict):
    backtesting_result.results["close_types"] = {}
print(backtesting_result.get_results_summary())

## Candlestick Chart with Signal Overlays

In [ ]:
import plotly.graph_objects as go

fig = backtesting_result.get_backtesting_figure()

# Overlay ATR compression bands
candles_df = backtesting_result.processed_data
if 'range_high' in candles_df.columns:
    fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df['range_high'],
                             line=dict(color='rgba(255,165,0,0.6)', width=1, dash='dot'),
                             name='Range High (20)'))
    fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df['range_low'],
                             line=dict(color='rgba(255,165,0,0.6)', width=1, dash='dot'),
                             name='Range Low (20)'))
if 'ema_risk' in candles_df.columns:
    fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df['ema_risk'],
                             line=dict(color='rgba(255,0,0,0.5)', width=1),
                             name='Risk-Off EMA (20)'))

fig.update_layout(height=700, title="E1 Compression Breakout — Entries/Exits")
fig.show()

## PnL per Trade

In [ ]:
import plotly.express as px

executors_df = backtesting_result.executors_df
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

fig = px.scatter(
    executors_df,
    x="timestamp", y="net_pnl_quote",
    title="PnL per Trade",
    color="profitable",
    color_discrete_map={True: "green", False: "red"},
    hover_data=["filled_amount_quote", "side", "close_type"],
)
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")
fig.update_layout(
    plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
    font=dict(color="white"), showlegend=False,
    xaxis=dict(gridcolor="gray"), yaxis=dict(gridcolor="gray"),
)
fig.show()

## PnL Distribution

In [ ]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PnL Distribution',
                   color='profitable', color_discrete_map={True: 'green', False: 'red'},
                   nbins=30)
fig.add_vline(x=0, line_dash="dash", line_color="white")
fig.update_layout(plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
                  font=dict(color="white"))
fig.show()

## Cumulative PnL Over Time

In [ ]:
executors_df_sorted = executors_df.sort_values('close_timestamp')
executors_df_sorted['cum_pnl'] = executors_df_sorted['net_pnl_quote'].cumsum()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=executors_df_sorted['close_timestamp'],
    y=executors_df_sorted['cum_pnl'],
    mode='lines',
    line=dict(color='#00FF88', width=2),
    name='Cumulative PnL',
    fill='tozeroy',
    fillcolor='rgba(0,255,136,0.1)',
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Cumulative PnL (USDT)", height=400,
    plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
    font=dict(color="white"),
    xaxis=dict(gridcolor="gray"), yaxis=dict(gridcolor="gray"),
)
fig.show()

## Exit Type Breakdown

In [ ]:
exit_counts = executors_df['close_type'].value_counts().reset_index()
exit_counts.columns = ['close_type', 'count']
fig = px.bar(exit_counts, x='close_type', y='count', title='Exit Types',
             color='close_type')
fig.update_layout(plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
                  font=dict(color="white"), showlegend=False)
fig.show()
executors_df['close_type_str'] = executors_df['close_type'].apply(lambda x: x.name if hasattr(x, 'name') else str(x))
print(executors_df[['side', 'net_pnl_quote', 'close_type_str']].groupby(['side','close_type_str']).agg(
    count=('net_pnl_quote','count'), avg_pnl=('net_pnl_quote','mean')).round(2))

## ATR Compression Over Time

Shows when the compression trigger was active. High compression periods = more signal opportunities.

In [ ]:
if 'atr_pct' in candles_df.columns:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df['atr_pct'],
                             line=dict(color='cyan', width=1), name='ATR Percentile'))
    fig.add_hline(y=0.20, line_dash="dash", line_color="orange",
                  annotation_text="Compression threshold (20th pct)")
    fig.update_layout(title="ATR Percentile — Compression Trigger",
                      plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
                      font=dict(color="white"), height=300,
                      xaxis=dict(gridcolor="gray"), yaxis=dict(gridcolor="gray"))
    fig.show()
else:
    print("atr_pct not in processed_data — check controller _compute_features")